<h1 style="text-align: center;">Example of Medical Pipeline</h1>

## 🔍 Overview

In this notebook, we illustrate how to use **assetflow** to build a medical data processing pipeline based on data collected from patients over the course of a multi-month study.

## 📦 Imports

In [5]:
import polars as pl

## 🗂️ Data Description

The data have been simulated and are available in: <code>assetflow/data/simulated/generated</code>
    
They consist of **four tables**.

### `glucometers` Table

The `glucometers` table contains information about devices used to measure blood sugar levels (glucometers).  
It includes the following columns:

- **gm_id**: unique identifier of the glucometer  
- **model_name**: commercial name of the glucometer model  
- **manufacturer**: name of the company that manufactured the glucometer  
- **year**: year the glucometer model was released  
- **class**: class of glucometers (*Bluetooth*, *Optical*, *NFC*)

In [4]:
glucometers = pl.read_csv(r"../data/simulated/generated/glucometers.csv")
glucometers

gm_id,model_name,manufacturer,year,class
i64,str,str,i64,str
1,"""Contour Plus One""","""Ascensia""",2016,"""Bluetooth"""
2,"""Accu-Chek Guide""","""Roche""",2017,"""Bluetooth"""
3,"""FreeStyle Lite""","""Abbott""",2015,"""Optical"""
4,"""OneTouch Verio Flex""","""LifeScan""",2016,"""Bluetooth"""
5,"""GlucoMen Areo""","""A. Minarine""",2018,"""NFC"""


### `appointments` Table

The `appointments` table contains clinical measurements collected during patient medical visits over the course of the study.

It has a shape of **(1,647 rows × 8 columns)** and includes the following fields:

- **date**: date and time of the medical appointment  
- **patient_id**: unique identifier of the patient  
- **height**: patient height (in centimeters)  
- **weight**: patient weight (in kilograms)  
- **gm_id**: identifier of the glucometer used during the appointment  
- **glycemia**: measured blood glucose level  
- **bp_sys**: systolic blood pressure  
- **bp_dia**: diastolic blood pressure  

Each row corresponds to a single appointment for a given patient.


In [10]:
appointments = pl.read_csv(r"../data/simulated/generated/appointments.csv")
appointments

date,patient_id,height,weight,gm_id,glycemia,bp_sys,bp_dia
str,i64,i64,f64,i64,f64,f64,f64
"""2019-12-05T14:30:00.000000""",85,186,70.3,4,101.155531,120.0,96.0
"""2018-01-04T16:15:00.000000""",90,170,70.3,1,94.235362,147.0,80.0
"""2017-04-05T17:15:00.000000""",96,192,94.0,4,98.376751,156.0,80.0
"""2019-02-05T09:45:00.000000""",61,175,69.3,4,64.726339,118.0,69.0
"""2018-12-01T09:45:00.000000""",49,175,81.0,3,104.809793,137.0,81.0
…,…,…,…,…,…,…,…
"""2016-12-06T13:15:00.000000""",31,166,98.8,5,87.934996,137.0,78.0
"""2017-12-03T13:00:00.000000""",28,189,92.5,1,76.546098,99.0,63.0
"""2020-06-06T14:30:00.000000""",5,159,84.7,5,87.936179,90.0,67.0


### `patients` Table

The `patients` table contains demographic information about the patients participating in the study.

The data are split into two CSV files (`1.csv` and `2.csv`), partitioned by **sex**.

It includes the following columns:

- **patient_id**: unique identifier of the patient  
- **sex**: biological sex of the patient (encoded as integers)  
- **name**: patient name  
- **birthdate**: patient date of birth  

This table provides static patient-level information and can be joined with other tables using `patient_id`.


In [12]:
patients = pl.read_csv(r"../data/simulated/generated/patients/*.csv")
patients

patient_id,sex,name,birthdate
i64,i64,str,str
1,1,"""RAYMOND""","""1931-11-28"""
4,1,"""CLAUDE""","""1971-06-16"""
8,1,"""DOMINIQUE""","""1959-04-07"""
9,1,"""ROBERT""","""1958-10-12"""
10,1,"""MORGAN""","""1984-04-05"""
…,…,…,…
82,2,"""MARIE""","""1946-09-02"""
86,2,"""JACQUELINE""","""1953-02-28"""
91,2,"""GHISLAINE""","""1955-04-08"""


### `glycemia` Table

The `glycemia` table contains longitudinal blood glucose measurements recorded outside of scheduled appointments.

The data are partitioned by **year**, with one CSV file per year (e.g. `2015.csv`, `2016.csv`, …).

It includes the following columns:

- **patient_id**: unique identifier of the patient  
- **date**: timestamp of the glycemia measurement  
- **gm_id**: identifier of the glucometer used  
- **glycemia**: measured blood glucose level  

This table allows fine-grained temporal analysis of glycemia evolution over time.

In [13]:
glycemia = pl.read_csv(r"../data/simulated/generated/glycemia/*.csv")
glycemia

patient_id,date,gm_id,glycemia
i64,str,i64,f64
17,"""2015-09-04T08:00:00.000000""",3,100.699729
17,"""2015-09-04T14:06:00.000000""",3,93.59373
17,"""2015-09-04T19:56:00.000000""",3,93.753316
17,"""2015-09-05T07:50:00.000000""",3,70.835508
17,"""2015-09-05T14:04:00.000000""",3,118.631929
…,…,…,…
48,"""2020-02-29T14:03:00.000000""",1,90.397269
48,"""2020-02-29T20:07:00.000000""",1,85.896898
48,"""2020-03-01T07:49:00.000000""",1,101.559777


## 📍 Creation of source Assets